## Day 2 - Baseline LightGBM Model

In [39]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sklearn.metrics import  accuracy_score 
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

from sklearn.model_selection import train_test_split
import lightgbm as lgb

In [40]:
df = pd.read_csv("fusion_pipeline_output_consolidated.csv", parse_dates=["timestamp"])

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (10000, 19)


,timestamp,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type,shift,production_load,power_quality_index,maintenance_window,operator_experience,wear_load_index,temperature_gap,power_stress_score
0,2025-01-01 00:00:00,1,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure,Night,0.749816,0.906046,0,2,0.000000,10.5,4.021226
1,2025-01-01 01:00:00,2,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure,Night,0.980286,0.899937,0,4,2.940857,10.5,4.632925
2,2025-01-01 02:00:00,3,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure,Night,0.892798,0.876423,0,5,4.463988,10.4,6.104700
3,2025-01-01 03:00:00,4,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure,Night,0.839463,0.941090,0,5,5.876244,10.4,2.326945
4,2025-01-01 04:00:00,5,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure,Night,0.662407,0.921494,0,3,5.961667,10.5,3.140255


In [41]:
target_column = "Target"
print(df[target_column].value_counts())

Target
0    9661
1     339
Name: count, dtype: int64


In [42]:
def evaluate_model(y_true, y_pred):

    print("Accuracy :", round(accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(y_true, y_pred), 4))
    print("Recall   :", round(recall_score(y_true, y_pred), 4))
    print("F1 Score :", round(f1_score(y_true, y_pred), 4))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

In [43]:
drop_cols = [target_column, "Failure Type", "timestamp", "Product ID","UDI"]
X = df.drop(columns=drop_cols)
y = df[target_column]
print("Feature Shape:", X.shape)
print("Target Shape :", y.shape)

Feature Shape: (10000, 14)
Target Shape : (10000,)


In [44]:
X = pd.get_dummies(X, columns=["Type", "shift"], drop_first=True)

print("Columns after encoding:")
print(X.columns.tolist())

Columns after encoding:
['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'production_load', 'power_quality_index', 'maintenance_window', 'operator_experience', 'wear_load_index', 'temperature_gap', 'power_stress_score', 'Type_L', 'Type_M', 'shift_Morning', 'shift_Night']


In [45]:
X.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", col) for col in X.columns]
print("Cleaned columns:")
print(X.columns.tolist())

Cleaned columns:
['Air_temperature_K_', 'Process_temperature_K_', 'Rotational_speed_rpm_', 'Torque_Nm_', 'Tool_wear_min_', 'production_load', 'power_quality_index', 'maintenance_window', 'operator_experience', 'wear_load_index', 'temperature_gap', 'power_stress_score', 'Type_L', 'Type_M', 'shift_Morning', 'shift_Night']


In [46]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print("Training Size:", X_train.shape)
print("Testing Size :", X_test.shape)

Training Size: (8000, 16)
Testing Size : (2000, 16)


## Baseline model

In [47]:
model = lgb.LGBMClassifier(random_state=42, verbose=-1)
model.fit(X_train, y_train)
print("Model trained")

Model trained


In [48]:
y_pred = model.predict(X_test)
evaluate_model(y_test, y_pred)

Accuracy : 0.99
Precision: 0.9286
Recall   : 0.7647
F1 Score : 0.8387

Confusion Matrix:
[[1928    4]
 [  16   52]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1932
           1       0.93      0.76      0.84        68

    accuracy                           0.99      2000
   macro avg       0.96      0.88      0.92      2000
weighted avg       0.99      0.99      0.99      2000



In [49]:
print("Training Size:", X_train.shape)
print("Testing Size :", X_test.shape)

print("Recall   :", round(recall_score(y_test, y_pred), 4))
print("F1 Score :", round(f1_score(y_test, y_pred), 4))

Training Size: (8000, 16)
Testing Size : (2000, 16)
Recall   : 0.7647
F1 Score : 0.8387


## Class weight balanced

In [50]:
weighted_model = lgb.LGBMClassifier(random_state=42, verbose=-1, class_weight="balanced")


In [51]:
weighted_model.fit(X_train, y_train)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [52]:
y_pred_weighted = weighted_model.predict(X_test)

In [53]:
evaluate_model(y_test, y_pred_weighted)

Accuracy : 0.9895
Precision: 0.8507
Recall   : 0.8382
F1 Score : 0.8444

Confusion Matrix:
[[1922   10]
 [  11   57]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.85      0.84      0.84        68

    accuracy                           0.99      2000
   macro avg       0.92      0.92      0.92      2000
weighted avg       0.99      0.99      0.99      2000



In [56]:
comparison_df = pd.DataFrame({
    "Metric": ["Recall", "Precision", "F1 Score"],
    "Baseline Model": baseline_scores,
    "Weighted Model": weighted_scores
})
print(comparison_df)

      Metric  Baseline Model  Weighted Model
0     Recall        0.764706        0.838235
1  Precision        0.928571        0.850746
2   F1 Score        0.838710        0.844444
